<a href="https://colab.research.google.com/github/NSCC-ITC-Fall2025-DBAS5115-700-MCr/assignment-2-emppacs/blob/main/BonusAssign2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2: Exploratory Data Analysis and Preprocessing for Predictive Modeling
Heart Disease Risk Prediction: Binary Classification

**Dataset:** [Heart_Disease]("https://www.kaggle.com/code/poornimawari/heart-disease-uci")

**Model Used:** Random Forest Classifier

**Target:** Cardiac Risk (Binary: 0 = No Risk, 1 = At Risk)

**Goal:** The objective of this analysis is to develop a high-accuracy screening tool using clinical patient data. This notebook demonstrates a full end-to-end Machine Learning pipeline—from Exploratory Data Analysis (EDA) to hyperparameter optimization.


# **1. Import Key Libraries**

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import missingno as msno
!pip install ydata_profiling

# Machine Learning Tools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

ModuleNotFoundError: No module named 'numpy'

# **2. Data Loading**

In [ ]:
# Loading the UCI Heart Disease dataset
df = pd.read_csv("heart_disease_uci.csv")
print(f"Data Shape: {df.shape}")
df.head()

   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

                thal  num  
0       fixed defect    0  
1             normal    2  
2  reversable defect    1  
3             normal    0  
4             normal    0  


# **3. Exploratory Data Analysis (EDA)**

In [ ]:
# Check for Missing Values
print("Missing Values per Column:")
print(df.isnull().sum())

# Visualizing Missingness
msno.matrix(df)
plt.title("Missing Data Map")
plt.show()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    object 
 3   dataset   920 non-null    object 
 4   cp        920 non-null    object 
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    object 
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    object 
 13  ca        309 non-null    float64
 14  thal      434 non-null    object 
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(8)
memory usage: 115.1+ KB
None



Data Shape: (920, 16)


***Visualization***

In [1]:
# Outlier Visualization (Box Plots)
numeric_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
plt.figure(figsize=(12, 6))
sns.boxplot(data=df[numeric_cols])
plt.title("Identifying Outliers in Numeric Features")
plt.show()

# Age Distribution by Heart Disease Presence
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='age', hue='num', multiple='stack', kde=True)
plt.title("Age Distribution vs. Disease Severity")
plt.show()

# Interactive Density Heatmap (Age vs Chol)
fig = px.density_heatmap(
    df, x='age', y='chol', facet_col='num',
    title='Density of Age vs Chol by Heart Disease Severity',
    template='plotly_dark'
)
fig.show()

NameError: name 'plt' is not defined

# **4. Data Preprocessing & Pipeline**

In [ ]:
# Convert 'num' to binary target (0 = No Disease, 1 = Disease Risk)
df['num'] = df['num'].apply(lambda x: 0 if x == 0 else 1)

X = df.drop(['num', 'id'], axis=1)
y = df['num']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Lists
NUMERIC_FEATURES = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
CATEGORICAL_FEATURES = ['sex', 'dataset', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

# Ensure categoricals are strings
X_train[CATEGORICAL_FEATURES] = X_train[CATEGORICAL_FEATURES].astype(str)
X_test[CATEGORICAL_FEATURES] = X_test[CATEGORICAL_FEATURES].astype(str)

# Preprocessing Pipelines
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, NUMERIC_FEATURES),
    ('cat', cat_pipe, CATEGORICAL_FEATURES)
])

# Final Pipeline
clf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=150, random_state=42))
])

# **5. Training and Results**

In [3]:
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

NameError: name 'clf' is not defined

# **6. Feature Importance**

In [ ]:
importances = clf.named_steps['classifier'].feature_importances_
# (Logic to extract feature names from preprocessor)
# ... plotting your bar chart ...
plt.title('Top 10 Feature Importances')
plt.show()

# **7. Hyperparameter Tuning (Grid Search)**

In [ ]:
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [8, 12, None],
    'classifier__min_samples_split': [5, 10],
}

grid_search = GridSearchCV(clf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Tuned Accuracy: {accuracy_score(y_test, grid_search.predict(X_test)):.4f}")